# Notebook for Loading and Plotting Degradation Data
#### Created in October 2022 by Dr Niall Kirkaldy for use with data in the Zenodo repository: 10.5281/zenodo.7235858
#### Updated in October 2023 by Dr Niall Kirkaldy for use with data in the Zenodo repository: 10.5281/zenodo.10637534
-------------------------

This notebook was made to aid in the further use of the lithium-ion battery cycle ageing data saved in the Zenodo repository 10.5281/zenodo.10637534 and detailed in the related publication in the Journal of Power Sources 10.1016/j.jpowsour.2024.234185 . Details of the license agreements can be found on these links.

This notebook was written in python 3.8.3 and requires pandas (1.0.5) and matplotlib (3.2.2) libraries.

---------------------------

## How-to:

----------------------

Prior to running the code in this notebook, you should download the data from Zenodo via the link above and unzip the folder. You will be asked to enter the directory where the data is saved; this should include the name of the (unzipped) folder called 'Battery Degradation Data'. If this notebook is saved in the same folder as the data, the input box can be left blank.

e.g., 'C:\Users\Niall\Imperial College London\LGM50 dataset - ME - General'

No other input should be required. Running the subsequent cells will load the data into a multi-level dictionary and plot some useful variables. There is quite a lot of data, so can take some time to load. If you only want data from a specific experiment, edit the top-level loop to only iterate over desired values (e.g. ['expt 1', 'expt 4']).

--------------------------

#### The structure of the data dictionary (called 'data_dict') is as follows:

    data_dict is a dictionary object with five keys: ['expt 1', 'expt 2,2', 'expt 3', 'expt 4', 'expt 5'].

    1. Each key has 3 sub-dictionaries: ['Main', 'Ageing', 'Timeseries'], which contain the different types of data as described in the repository on Zenodo. In all cases, the keys in these dictionaries are the cell IDs (i.e. 'A', 'B', etc.). These can also be viewed in the 'experiment_metadata' excel file.
    
        1.1 'Main' contains a dictionary of pandas DataFrame objects (one entry for each cell tested). These DataFrames house most of the important degradation data, such as charge throughput, capacity fade, resistance increase, and the degradation modes determined from OCV-fitting of the C/10 discharge data from each RPT. 
        E.g. data_dict['expt 1']['Main']['A'] is a DataFrame with the data for cell A of experiment 1 (0-30% SoC cycling). 
        This has the general form: data_dict[EXPERIMENT]['Main'][CELL_ID]
        
        1.2 'Ageing' similarly contains a dictionary of pandas DataFrame objects (one entry for each cell tested). These DataFrames house summary variables from the ageing sets, such as average temperature during discharge, etc. Each row corresponds to a different ageing set. Alternatively, there are similar variables available on a per-cycle basis in the same folder (not loaded here). 
        E.g. data_dict['expt 5']['Ageing']['H'] is a DataFrame with operating variables which have been averaged over each ageing set for cell 'H' of experiment 5 (0-100% SoC cycling). 
        This has the general form: data_dict[EXPERIMENT]['Ageing'][CELL_ID]
        
        1.3 'Timeseries' contains the raw timeseries data for each cell at each reference performance test (RPT).As above, there is one entry per cell. However, each entry contains a list with each RPT for that cell. For each RPT there is a further list which contains raw data for each of the multiple sub-tests of the RPT (saved as pandas DataFrames). 
        E.g. data_dict['expt 1']['Timeseries']['E'][4][0] is a DataFrame with the C/10 discharge data at RPT 4 for cell E in experiment 1 (0-30% SoC cycling). 
        The general form is data_dict[EXPERIMENT]['Timeseries'][CELL_ID][RPT_NUMBER][SUB_TEST]. Details on the subtests can be found in the publication in the Journal of Power Sources and in the Zenodo repository.
        

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib notebook

In [ ]:
# You can leave blank if this notebook is saved in root directory of data
root_filepath = input('File directory of data = ')

In [ ]:
if len(root_filepath) > 0:
    root_filepath += '/'
main_data_path = 'Summary Data'

# Read metadata from excel file
meta_data_dict = {}
expt_list = ['expt 1', 'expt 2,2', 'expt 3', 'expt 4', 'expt 5']
for expt in expt_list:
    meta_data = pd.read_excel(f'{root_filepath}experiment_metadata.xlsx', sheet_name=f'{expt}')
    meta_data.set_index('Cell', inplace=True, drop=False)
    meta_data_dict[expt] = meta_data

# Define colours for plotting data later.  
colour_temp_dict = {10:'C0', 25:'C1', 40:'C3'}

In [ ]:
data_dict = {}
# Edit the top-level 'for' loop if you don't want to load data from all expts
for expt in meta_data_dict.keys():
    long_name = meta_data_dict[expt].loc['A', 'Expt long name']
    SoC_range = meta_data_dict[expt].loc['A', 'SoC range']
    main_data = {}
    age_set_data = {}
    timeseries_data = {}
    # Edit the next 'for' loop if you don't want to load data from all cells (within current expt)
    for cell in meta_data_dict[expt].index.values:
        temp_val = meta_data_dict[expt].loc[cell, 'Temp']
        cell_ID = meta_data_dict[expt].loc[cell, 'Cell']
        main_data[cell_ID] = pd.read_csv(f'{root_filepath}{long_name}/{main_data_path}/Performance Summary/Expt {expt[5:]} - cell {cell_ID} ({temp_val}degC) - Processed Data.csv', index_col=0)
        age_set_data[cell_ID] = pd.read_csv(f'{root_filepath}{long_name}/{main_data_path}/Ageing Sets Summary/Summary per Set/Expt {expt[5:]} - cell {cell_ID} - set_data.csv', index_col=0)
        cell_timeseries_data = []
        for rpt in range(len(main_data[cell_ID])):
            #Even-numbered RPTs have 4 sub-tests, whereas odd-numbered RPTs have 3.
            if rpt % 2 == 0:
                CC_0p1C = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/0.1C Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - 0.1C discharge data.csv')
                CC_0p5C = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/0.5C Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - 0.5C discharge data.csv')
                GITT_25p = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/GITT Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - 25-pulse GITT 0.5C discharge data.csv')
                GITT_5p = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/GITT Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - 5-pulse GITT 0.5C discharge data.csv')
                cell_timeseries_data.append([CC_0p1C, CC_0p5C, GITT_25p, GITT_5p])
            else:
                CC_0p1C = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/0.1C Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - 0.1C discharge data.csv')
                hybrid_pulse_0p5C = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/Hybrid CC-Pulse Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - Hybrid CC-Pulse 0.5C discharge data.csv')
                hybrid_pulse_1C = pd.read_csv(f'{root_filepath}{long_name}/Processed Timeseries Data/Hybrid CC-Pulse Voltage Curves/cell {cell_ID}/Expt {expt[5:]} - cell {cell_ID} - RPT{rpt} - Hybrid CC-Pulse 1C discharge data.csv')
                cell_timeseries_data.append([CC_0p1C, hybrid_pulse_0p5C, hybrid_pulse_1C])
        timeseries_data[cell_ID] = cell_timeseries_data
    data_dict[expt] = {'Main':main_data, 'Ageing':age_set_data, 'Timeseries':timeseries_data}

## View dataframes

An example of each of the types of data from the 'Extracted Data' folder, as detailed above.

Main: Summary data for each cell.

Timeseries: raw data from each section of each RPT.

Ageing: Summary variables from the cycle ageing (for each cell at each ageing set).

In [ ]:
data_dict['expt 4']['Main']['C']

In [ ]:
data_dict['expt 5']['Timeseries']['A'][0][0]

In [ ]:
data_dict['expt 1']['Ageing']['K']

## Plot some of the data

Example plots for each of the types of data from the 'Summary Data' and 'Processed Timeseries Data' folders (timeseries, main, ageing).

In [ ]:
fig_timeseries, ax = plt.subplots(2, figsize=(8, 10), sharex=True)

# Edit the value of 'expt' if you want to plot for a different expt.
expt = 'expt 3'
# Edit the value of 'cell' if you want to plot for a different cell.
cell = 'D'

x_variable_1 = 'Charge (mA.h)'
y_variable_1 = 'Voltage (V)'
x_variable_2 = x_variable_1
y_variable_2 = y_variable_1

for rpt in range(len(data_dict[expt]['Timeseries'][cell])):
    # Plot C/10 discharge data
    data_dict[expt]['Timeseries'][cell][rpt][0].plot(x_variable_1, y_variable_1, ax=ax[0], c='k', alpha=(1-rpt/len(data_dict[expt]['Timeseries'][cell])), label=f'RPT {rpt}')
    if rpt % 2 == 0:
        # Some RPT data is only available on even or odd numbered RPTs (e.g. GITT)
        data_dict[expt]['Timeseries'][cell][rpt][2].plot(x_variable_2, y_variable_2, ax=ax[1], c='k', alpha=(1-rpt/len(data_dict[expt]['Timeseries'][cell])), label=f'RPT {rpt}')
    
#ax[0].set_xlabel(x_variable_1)
ax[0].set_ylabel(y_variable_1)
ax[1].set_xlabel(x_variable_2)
ax[1].set_ylabel(y_variable_2)

age_temp = meta_data_dict[expt].loc[cell, 'Temp']
age_SoC = meta_data_dict[expt].loc[cell, 'SoC range']
ax[0].set_title(f'V vs Q data for {expt} cell {cell} (cycled at {age_SoC}% SoC / {age_temp} degC)')

fig_timeseries.tight_layout()

In [ ]:
fig_ageing, ax = plt.subplots(2, sharex=True)

# Edit the value of 'expt' if you want to plot for a different expt.
expt = 'expt 2,2'
# Can plot as a function of other x-variables (e.g. 'Ageing Cycles', or 'Days of degradation').
x_variable = 'Charge Throughput [A h]'
y_variable_1 = 'C/10 Capacity [mA h]'
y_variable_2 = '0.1s Resistance [Ohms]'

for cell in meta_data_dict[expt].index.values:
    temp_val = meta_data_dict[expt].loc[cell, 'Temp']
    data_dict[expt]['Main'][cell].plot(x_variable, y_variable_1, ax=ax[0], c=colour_temp_dict[temp_val], marker='o')
    # Resistance values from GITT, so only avaiable on even-numbered RPTs
    df_slice = data_dict[expt]['Main'][cell].loc[range(0,len(data_dict[expt]['Main'][cell]), 2), :]
    df_slice.plot(x_variable, y_variable_2, ax=ax[1], c=colour_temp_dict[temp_val], marker='o')

ax[1].set_xlabel(x_variable)
ax[0].set_ylabel(y_variable_1)
ax[1].set_ylabel(y_variable_2)
ax[0].grid(True)
ax[1].grid(True)
SOC_val = meta_data_dict[expt].loc[cell, 'SoC range']
ax[0].set_title(f'Ageing Data for cells cycled at {SOC_val}% SoC')

colours = colour_temp_dict
labels = [f'{temp_value} degC' for temp_value in colour_temp_dict.keys()]
handles = [plt.Rectangle((0,0),1,1, color=colours[temp_value]) for temp_value in colour_temp_dict.keys()]
ax[0].legend(handles, labels, loc='best')
ax[1].legend(handles, labels, loc='best')

fig_ageing.tight_layout()

In [ ]:
fig_DMA, ax = plt.subplots(6, figsize=(8,10), sharex=True)

# Edit the value of 'expt' if you want to plot for a different expt.
expt = 'expt 1'
# Can plot as a function of other x-variables (e.g. 'Ageing Cycles', or 'Days of degradation').
x_variable = 'Charge Throughput [A h]'

for cell in meta_data_dict[expt].index.values:
    temp_val = meta_data_dict[expt].loc[cell, 'Temp']
    # Take x-axis values from 'Main'
    x_values = data_dict[expt]['Main'][cell].loc[:,x_variable]
    # Plot all available DMA variables (on separate axes)
    for number, key in enumerate(['SoH', 'LAM PE', 'LAM NE', 'LAM NE_Gr', 'LAM NE_Si', 'LLI']):
        ax[number].plot(x_values, data_dict[expt]['Main'][cell].loc[:,key], c=colour_temp_dict[temp_val], marker='o')
        ax[number].set_ylabel(key)
        ax[number].grid(True)

ax[5].set_xlabel(x_variable)
SOC_val = meta_data_dict[expt].loc[cell, 'SoC range']
ax[0].set_title(f'DMA analysis for cells cycled at {SOC_val}% SoC')

colours = colour_temp_dict
labels = [f'{temp_value} degC' for temp_value in colour_temp_dict.keys()]
handles = [plt.Rectangle((0,0),1,1, color=colours[temp_value]) for temp_value in colour_temp_dict.keys()]
ax[0].legend(handles, labels, loc='best')

fig_DMA.tight_layout()